# Silver Layer — NSW Air Quality

Reads `workspace.aq_bronze.observations` (3,998,208 rows), flattens the nested
Parameter struct, constructs a correct timestamp, validates every row, and splits
clean rows from rejected ones.

**Reconciliation identity:** `bronze = silver + quarantine + duplicates removed`

| Output | Contents |
|---|---|
| `workspace.aq_silver.dim_station` | Reference: stations |
| `workspace.aq_silver.dim_parameter` | Reference: parameters |
| `workspace.aq_silver.fact_observation` | Clean readings |
| `workspace.aq_quarantine.observations` | Rejected rows, with reason |

In [0]:
from pyspark.sql import functions as F
import requests

BASE = "https://data.airquality.nsw.gov.au/api/Data"

## 1. Dimension tables

Small reference tables the fact table joins to.

Data quality handling:
- Source region names have inconsistent capitalisation ("Sydney North-west" and
  "Sydney north-west" both occur). Normalised with `initcap(lower())` so regional
  grouping does not split one region into two.
- Two Sydney-region entries are regional aggregates rather than physical stations,
  identified by null coordinates. Excluded to prevent double-counting.

In [0]:
sites = requests.get(f"{BASE}/get_SiteDetails").json()

stations = [s for s in sites
            if s["Region"].lower().startswith("sydney")
            and s["Latitude"] is not None
            and s["Longitude"] is not None]

dim_station = (spark.createDataFrame(stations)
    .select(
        F.col("Site_Id").cast("int").alias("site_id"),
        F.initcap(F.lower(F.col("SiteName"))).alias("site_name"),
        F.initcap(F.lower(F.col("Region"))).alias("region"),
        F.col("Latitude").cast("double").alias("latitude"),
        F.col("Longitude").cast("double").alias("longitude"),
    ))

dim_station.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.aq_silver.dim_station")

print(dim_station.count(), "stations")
dim_station.show(30, truncate=False)

24 stations
+-------+-----------------+-----------------+--------------+--------------+
|site_id|site_name        |region           |latitude      |longitude     |
+-------+-----------------+-----------------+--------------+--------------+
|33     |Randwick         |Sydney East      |-33.93175     |151.24278     |
|39     |Rozelle          |Sydney East      |-33.86433     |151.16395     |
|70     |Lindfield        |Sydney East      |-33.78113     |151.1509      |
|107    |Liverpool        |Sydney South-west|-33.93132     |150.90727     |
|171    |Bringelly        |Sydney South-west|-33.91766     |150.76192     |
|190    |Chullora         |Sydney East      |-33.89156     |151.0461      |
|206    |Earlwood         |Sydney East      |-33.91619     |151.13577     |
|573    |Richmond         |Sydney North-west|-33.61641     |150.74731     |
|574    |Bargo            |Sydney South-west|-34.30621     |150.58061     |
|760    |St Marys         |Sydney North-west|-33.79512     |150.76677     |


In [0]:
params = requests.get(f"{BASE}/get_ParameterDetails").json()

dim_parameter = (spark.createDataFrame(params)
    .select(
        F.col("ParameterCode").alias("parameter_code"),
        F.col("ParameterDescription").alias("parameter_name"),
        F.col("Units").alias("units"),
        F.col("UnitsDescription").alias("units_description"),
    )
    .dropDuplicates(["parameter_code", "units"]))

dim_parameter.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.aq_silver.dim_parameter")

print(dim_parameter.count(), "parameter definitions")

29 parameter definitions


## 2. Flatten and type

Three transformations:
1. Nested `Parameter` struct flattened into columns
2. Explicit type on every column
3. Timestamp constructed from `Date` + `Hour`

**The `Hour` column is 1-BASED.** `Hour=1` carries `HourDescription` "12 am - 1 am",
so hour 1 is midnight. Subtracting 1 before adding the interval prevents a
one-hour shift across the entire dataset.

In [0]:
flat = (spark.table("workspace.aq_bronze.observations")
    .select(
        F.col("Site_Id").cast("int").alias("site_id"),
        F.col("Parameter.ParameterCode").alias("parameter_code"),
        F.col("Parameter.Units").alias("units"),
        F.col("Parameter.Frequency").alias("frequency"),
        F.to_date("Date").alias("obs_date"),
        F.col("Hour").cast("int").alias("hour_1based"),
        F.col("Value").cast("double").alias("value"),
        F.col("AirQualityCategory").alias("aqi_category"),
        F.col("_source_file"),
        F.col("_ingested_at"),
    )
    .withColumn("obs_hour", F.col("hour_1based") - 1)
    .withColumn("obs_time",
        F.to_timestamp(F.col("obs_date")) + F.expr("INTERVAL 1 HOUR") * F.col("obs_hour"))
    .withColumn("year",  F.year("obs_date"))
    .withColumn("month", F.month("obs_date")))

flat.select("site_id", "parameter_code", "obs_date",
            "hour_1based", "obs_time", "value").show(3, truncate=False)

+-------+--------------+----------+-----------+-------------------+--------+
|site_id|parameter_code|obs_date  |hour_1based|obs_time           |value   |
+-------+--------------+----------+-----------+-------------------+--------+
|573    |OZONE         |2020-01-01|1          |2020-01-01 00:00:00|1.905025|
|573    |OZONE         |2020-01-01|2          |2020-01-01 01:00:00|NULL    |
|573    |OZONE         |2020-01-01|3          |2020-01-01 02:00:00|1.77135 |
+-------+--------------+----------+-----------+-------------------+--------+
only showing top 3 rows


### CHECKPOINT — stop and look at the output above

Where `hour_1based` is 1, `obs_time` **must** end in `00:00:00` on the same date.

| What you see | Meaning |
|---|---|
| `2020-01-01 00:00:00` for hour 1 | Correct. Continue |
| `2020-01-01 01:00:00` for hour 1 | Off-by-one still present. Stop and fix |

Every daily average, hourly profile and trend built tomorrow inherits this.

## 3. Validation

Every row is tagged with a rejection reason or NULL. Rows are never deleted —
failures go to the quarantine table with the reason recorded, so counts always
reconcile and every rejection is auditable.

Expected: 10–14% null values, reflecting station downtime and instruments not
present at every site.

In [0]:
tagged = (flat
    .withColumn("reject_reason", F.expr("""
        CASE
          WHEN site_id     IS NULL                 THEN 'null_site'
          WHEN obs_date    IS NULL                 THEN 'null_date'
          WHEN hour_1based < 1 OR hour_1based > 24 THEN 'hour_out_of_range'
          WHEN value       IS NULL                 THEN 'null_value'
          WHEN value       < -20                   THEN 'implausible_negative'
          ELSE NULL
        END
    """))
    .withColumn("below_detection", F.expr("value < 0"))
    .withColumn("value_clean",     F.expr("GREATEST(value, 0)")))

tagged.groupBy("reject_reason").count().orderBy(F.desc("count")).show()
tagged.groupBy("below_detection").count().show()

+-------------+-------+
|reject_reason|  count|
+-------------+-------+
|         NULL|3507029|
|   null_value| 491179|
+-------------+-------+

+---------------+-------+
|below_detection|  count|
+---------------+-------+
|          false|3345570|
|           NULL| 491179|
|           true| 161459|
+---------------+-------+



## 4. Split into silver and quarantine

**Grain of `fact_observation`:** one row per site, per parameter, per hour.

In [0]:
(tagged.filter(F.col("reject_reason").isNotNull())
   .withColumn("quarantined_at", F.current_timestamp())
   .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable("workspace.aq_quarantine.observations"))

print("quarantined:", spark.table("workspace.aq_quarantine.observations").count())

quarantined: 491179


In [0]:
clean = tagged.filter(F.col("reject_reason").isNull()).drop("reject_reason")
before_dedupe = clean.count()

(clean.dropDuplicates(["site_id", "parameter_code", "obs_time"])
   .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable("workspace.aq_silver.fact_observation"))

after_dedupe = spark.table("workspace.aq_silver.fact_observation").count()

print(f"before dedupe : {before_dedupe:,}")
print(f"after dedupe  : {after_dedupe:,}")
print(f"duplicates    : {before_dedupe - after_dedupe:,}")

before dedupe : 3,507,029
after dedupe  : 3,507,029
duplicates    : 0


## 5. Reconciliation

`bronze = silver + quarantine + duplicates removed`

Must balance exactly. A mismatch is a bug, not a rounding difference.

In [0]:
bronze = spark.table("workspace.aq_bronze.observations").count()
silver = spark.table("workspace.aq_silver.fact_observation").count()
quar   = spark.table("workspace.aq_quarantine.observations").count()
dupes  = before_dedupe - after_dedupe

print(f"bronze     : {bronze:,}")
print(f"silver     : {silver:,}")
print(f"quarantine : {quar:,}")
print(f"duplicates : {dupes:,}")
print(f"BALANCED   : {bronze == silver + quar + dupes}")

bronze     : 3,998,208
silver     : 3,507,029
quarantine : 491,179
duplicates : 0
BALANCED   : True


In [0]:
violations = spark.sql("""
SELECT site_id, parameter_code, obs_time, count(*) AS n
FROM workspace.aq_silver.fact_observation
GROUP BY 1, 2, 3 HAVING count(*) > 1
""").count()

print("grain violations:", violations)

grain violations: 0


## 6. Quarantine profile

Screenshot this output. "N% of raw readings quarantined, broken down by cause"
is a far stronger README line than "cleaned the data".

In [0]:
spark.sql("""
SELECT reject_reason,
       count(*) AS rows,
       round(100.0 * count(*) /
             (SELECT count(*) FROM workspace.aq_bronze.observations), 2) AS pct_of_bronze
FROM workspace.aq_quarantine.observations
GROUP BY 1 ORDER BY 2 DESC
""").show()

+-------------+------+-------------+
|reject_reason|  rows|pct_of_bronze|
+-------------+------+-------------+
|   null_value|491179|        12.28|
+-------------+------+-------------+



In [0]:
spark.sql("""
SELECT parameter_code,
       count(*) AS total,
       sum(CASE WHEN below_detection THEN 1 ELSE 0 END) AS below_detection,
       round(100.0 * sum(CASE WHEN below_detection THEN 1 ELSE 0 END) / count(*), 2) AS pct_bdl
FROM workspace.aq_silver.fact_observation
GROUP BY 1 ORDER BY 4 DESC
""").show()

+--------------+------+---------------+-------+
|parameter_code| total|below_detection|pct_bdl|
+--------------+------+---------------+-------+
|         PM2.5|883867|          80774|   9.14|
|           NO2|858005|          55470|   6.46|
|         OZONE|864833|          13604|   1.57|
|          PM10|900324|          11611|   1.29|
+--------------+------+---------------+-------+

